In [4]:
from sklearn.model_selection import train_test_split
import pandas as pd
fish_full = pd.read_csv('data/fish.csv')
fish_full = fish_full.to_numpy()
fish_weight = pd.read_csv('data/fish_weight.csv')
train_input, test_input, train_target, test_target = \
    train_test_split(
        fish_full,
        fish_weight,
        random_state=42
    )

### sklearndmfh 다중회귀 분석하기

In [7]:
from sklearn.preprocessing import PolynomialFeatures

In [15]:
poly = PolynomialFeatures() # include_bias:False: 절편은 제외하고 특성 , degree = 2(차원)
# poly.fit([[2, 3]]) # 학습
# poly.transform([[2, 3]]) #변형
poly.fit_transform([[2, 3]])
# array([[절편., x., y., x^2., x*y., y^2.]])

array([[1., 2., 3., 4., 6., 9.]])

In [14]:
poly.get_feature_names_out()

array(['1', 'x0', 'x1', 'x0^2', 'x0 x1', 'x1^2'], dtype=object)

### 다항 특성 만들기

In [16]:
train_input.shape

(42, 3)

In [19]:
# train_input을 다항 특성 만들기
poly = PolynomialFeatures(include_bias=False)
train_poly = poly.fit_transform(train_input)
train_poly.shape

(42, 9)

In [20]:
poly.get_feature_names_out()

array(['x0', 'x1', 'x2', 'x0^2', 'x0 x1', 'x0 x2', 'x1^2', 'x1 x2',
       'x2^2'], dtype=object)

In [23]:
# test_input을 다항 특성 만들기
test_poly = poly.transform(test_input) # test 데이터는 fit 하면 안됨
test_poly.shape

(14, 9)

In [24]:
from sklearn.linear_model import LinearRegression

In [28]:
lr = LinearRegression()
lr.fit(train_input, train_target)
print(f'Train: {lr.score(train_input, train_target)}')
print(f'Test: {lr.score(test_input, test_target)}')
# 과대적합이면서 차이가 5% 넘어감, 사용 불가

Train: 0.9559326821885706
Test: 0.8796419177546368


In [30]:
lr.fit(train_poly, train_target)
print(f'Train: {lr.score(train_poly, train_target)}')
print(f'Test: {lr.score(test_poly, test_target)}')
# 차이가 5% 넘어가지 않음, 좋은 평가

Train: 0.9903183436982124
Test: 0.9714559911594207


### 더 많은 특성 만들기

In [32]:
poly = PolynomialFeatures(include_bias=False, degree=5) # 5(차원)
poly.fit(train_input)
train_poly = poly.transform(train_input)
test_poly = poly.transform(test_input)

In [34]:
train_poly.shape
# 특성이 많아지면 속도가 저하 적당한 특성 갯수를 정하는 것이 중요

(42, 55)

In [37]:
lr.fit(train_poly, train_target)
print(f'Train: {lr.score(train_poly, train_target)}') # 거의 100% 좋은 평가
print(f'Test: {lr.score(test_poly, test_target)}')

Train: 0.9999999999997834
Test: -144.40494191100848


### 규제
모델이 과적합되게 학습하지 않고 일반성을 가질 수 있도록 규제(정규화) 하는 것

In [38]:
from sklearn.preprocessing import StandardScaler
ss = StandardScaler()
ss.fit(train_poly)

train_scaled = ss.transform(train_poly)
test_scaled = ss.transform(test_poly)

In [39]:
from sklearn.linear_model import Ridge

In [43]:
ridge = Ridge()
ridge.fit(train_scaled, train_target)
print(f'Train: {ridge.score(train_scaled, train_target)}') 
print(f'Test: {ridge.score(test_scaled, test_target)}') # -144.404 > 0.979로 많이 나아짐

Train: 0.9896101671037343
Test: 0.9790693977615373


In [48]:
train_score = []
test_score = []
alpha_list = [0.001, 0.01, 0.1, 1, 10, 100]

for alpha in alpha_list:
    ridge = Ridge(alpha=alpha)
    ridge.fit(train_scaled, train_target)
    train_score.append(ridge.score(train_scaled, train_target))
    test_score.append(ridge.score(test_scaled, test_target))
print(train_score)
print(test_score)

[0.993045525208813, 0.9917809981250554, 0.9903815817570367, 0.9896101671037343, 0.988728468997471, 0.9841843235774494]
[0.9569388961577475, 0.9800187936873267, 0.9827976465386898, 0.9790693977615373, 0.9725329582461569, 0.9627042641050291]


In [50]:
pd.DataFrame({
    'Train':train_score,
    'Test':test_score
})

,Train,Test
0,0.993046,0.956939
1,0.991781,0.980019
2,0.990382,0.982798
3,0.989610,0.979069
4,0.988728,0.972533
5,0.984184,0.962704
